# 🏦 Banking Streaming Pipeline — PySpark Notebook

**Pipeline:** Cribl DataGen → Kafka → **Spark Structured Streaming** → Apache Iceberg → Dell Data Lakehouse

| Cell | Description |
|------|-------------|
| 1    | Install packages & configure SparkSession |
| 2    | Define schemas (payments, fraud, customer) |
| 3    | Create Iceberg tables (DDL) |
| 4    | Stream 1 — raw payment transactions → Iceberg |
| 5    | Stream 2 — windowed payment aggregates → Iceberg |
| 6    | Stream 3 — fraud signals + AML + customer events → Iceberg |
| 7    | Monitor active streams |
| 8    | Ad-hoc Iceberg queries (time travel, compaction) |
| 9    | Stop all streams |

> **Tested on:** JupyterHub with PySpark kernel, DDLH, and Apache Zeppelin.  

In [ ]:
import subprocess

subprocess.run(["mkdir", "-p", "/home/jovyan/local_jars"])
subprocess.run(["tar", "-xzf", "/home/jovyan/workspace/spark_jars_hdfs.tar.gz", "-C", "/home/jovyan/local_jars"])

import glob
jars = glob.glob("/home/jovyan/local_jars/*.jar")
print(f"{len(jars)} JARs found")
for j in sorted(jars):
    print(" ", j)

In [ ]:
import glob
import os

JAR_DIR = "/home/jovyan/local_jars"
all_jar_paths = glob.glob(f"{JAR_DIR}/*.jar")

# Dedupe by file size (identical jars have identical byte size regardless of filename)
seen_sizes = {}
for path in all_jar_paths:
    size = os.path.getsize(path)
    base = os.path.basename(path)
    if size not in seen_sizes:
        seen_sizes[size] = path
    else:
        # Prefer the shorter filename (no groupId prefix) when sizes match
        if len(base) < len(os.path.basename(seen_sizes[size])):
            seen_sizes[size] = path

deduped_jars = sorted(seen_sizes.values())
print(f"Deduped to {len(deduped_jars)} jars (from {len(all_jar_paths)}):")
for j in deduped_jars:
    print(" ", os.path.basename(j))

In [ ]:
# ── Upload your manually-downloaded JARs to this environment first ──────────
# Recommended: create a folder like /home/jovyan/local_jars/ and upload all
# JARs there via the Jupyter file browser (or untar a bundle you uploaded).
#
# For Option B (S3 streaming checkpoints), also download these two and add
# them to the same folder — you already have hadoop-client-api/runtime
# 3.3.4, so match that version:
#   https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar
#   https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar
#
# Note: Ivy-resolved jars often appear twice — once plain (e.g.
# 'iceberg-aws-bundle-1.5.0.jar') and once with a groupId prefix (e.g.
# 'org.apache.iceberg_iceberg-aws-bundle-1.5.0.jar'). Both are byte-for-byte
# identical; this cell dedupes by FILE SIZE (more reliable than filename
# parsing), preferring the shorter filename when a duplicate is found.

import glob
import os

JAR_DIR = "/home/jovyan/local_jars"  # <-- change if you uploaded elsewhere

if not os.path.isdir(JAR_DIR):
    print(f"⚠️  {JAR_DIR} does not exist yet — create it and upload your JARs there.")
    deduped_jars = []
else:
    all_jar_paths = glob.glob(f"{JAR_DIR}/*.jar")

    # Dedupe by file size — identical jars have identical byte size
    # regardless of how Ivy named the file on disk.
    seen_sizes = {}
    for path in all_jar_paths:
        size = os.path.getsize(path)
        base = os.path.basename(path)
        if size not in seen_sizes:
            seen_sizes[size] = path
        elif len(base) < len(os.path.basename(seen_sizes[size])):
            seen_sizes[size] = path  # prefer the shorter (non-prefixed) filename

    deduped_jars = sorted(seen_sizes.values())
    print(f"📋 Found {len(all_jar_paths)} jar(s) in {JAR_DIR}, deduped to {len(deduped_jars)}:\n")
    for j in deduped_jars:
        size_mb = os.path.getsize(j) / (1024 * 1024)
        print(f"  {os.path.basename(j):55s} {size_mb:6.1f} MB")

    # Sanity check for the core jars we expect, including hadoop-aws for
    # Option B (S3 streaming checkpoints).
    expected_prefixes = [
        "iceberg-spark-runtime", "iceberg-aws-bundle", "spark-sql-kafka",
        "hadoop-aws", "aws-java-sdk-bundle",
    ]
    found_names = [os.path.basename(j) for j in deduped_jars]
    missing = [p for p in expected_prefixes if not any(p in n for n in found_names)]
    if missing:
        print(f"\n⚠️  Missing expected jar(s) starting with: {missing}")
    else:
        print("\n✅ All 5 core jars present (including hadoop-aws for S3 checkpoints).")

## Cell 1 — Install Packages & Build SparkSession

In [ ]:
import os

# ── Build --jars argument from the deduped local JAR list (Cell above) ──────
# Now includes hadoop-aws + aws-java-sdk-bundle, so Hadoop's generic
# FileSystem layer can resolve the "s3://" scheme for streaming
# checkpoints (separate code path from Iceberg's native S3FileIO,
# which only handles table DATA, not checkpoint metadata).
all_jars = ",".join(deduped_jars)
if not all_jars:
    raise RuntimeError(
        "No jars found — run the previous cell after uploading your jars."
    )
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {all_jars} pyspark-shell"
print(f"PYSPARK_SUBMIT_ARGS set with {len(deduped_jars)} local jars.")

# ── Config — your environment ────────────────────────────────────────────────
KAFKA_BROKERS   = "48.64.33.52:9092"

# Glue-compatible catalog — Dell DDLH managed metastore (Glue API proxy)
GLUE_ENDPOINT   = "http://managed-metastore.ddae.svc.cluster.local:8080/api/v1/glue"
GLUE_REGION     = "us-east-1"
GLUE_CATALOG_ID = "bnk_ice"
GLUE_ACCESS_KEY = os.environ["GLUE_ACCESS_KEY"]
GLUE_SECRET_KEY = os.environ["GLUE_SECRET_KEY"]

# Native S3 endpoint (Dell PowerScale), separate creds from Glue
S3_ENDPOINT    = "http://f910.uds.csc.poc.apj.lab:9020"
S3_ACCESS_KEY  = os.environ["S3_ACCESS_KEY"]
S3_SECRET_KEY  = os.environ["S3_SECRET_KEY"]

WAREHOUSE      = "s3://aidp-s3ds/warehouse/banking/warehouse"
# Checkpoints now live in S3 B) — durable across pod restarts.
# Requires hadoop-aws + aws-java-sdk-bundle on the classpath (added above)
# so Hadoop's FileSystem can resolve the "s3://" scheme.
CHECKPOINT     = "s3://aidp-s3ds/warehouse/banking/checkpoints"
CATALOG        = "banking_iceberg"
DB             = "banking"

# ── IMPORTANT: Iceberg's Java GlueCatalog client does NOT support
# spark.sql.catalog.X.client.access-key-id / secret-access-key — that
# property pair only works for the S3FileIO client, not the Glue client
# itself (confirmed: https://github.com/apache/iceberg/issues/10614).
# The Glue client falls back to the standard AWS SDK credential chain,
# which DOES read AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY env vars.
# We set those here, scoped to this process only, so they don't leak
# into the wider system.
os.environ["AWS_ACCESS_KEY_ID"]     = GLUE_ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = GLUE_SECRET_KEY
os.environ["AWS_REGION"]            = GLUE_REGION

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"]  = "WHEN_REQUIRED"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"]  = "WHEN_REQUIRED"

print(f"Kafka  : {KAFKA_BROKERS}")
print(f"Glue   : {GLUE_ENDPOINT} (catalogId={GLUE_CATALOG_ID}) — using AWS_* env vars")
print(f"S3     : {S3_ENDPOINT} — using catalog-scoped s3.access-key-id + hadoop-aws for checkpoints")
print(f"Catalog: {CATALOG}.{DB}")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BankingStreamingJob")
#    .master("local[2]")                              # remove for YARN / k8s cluster

    .config("spark.driver.extraJavaOptions",
            "-Daws.requestChecksumCalculation=WHEN_REQUIRED "
            "-Daws.responseChecksumValidation=WHEN_REQUIRED")
    .config("spark.executor.extraJavaOptions",
            "-Daws.requestChecksumCalculation=WHEN_REQUIRED "
            "-Daws.responseChecksumValidation=WHEN_REQUIRED")
    
    # ── Iceberg extensions ──────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    # ── Iceberg catalog → Glue-compatible catalog via Starburst proxy ───
    .config(f"spark.sql.catalog.{CATALOG}",
            "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG}.catalog-impl",
            "org.apache.iceberg.aws.glue.GlueCatalog")
    .config(f"spark.sql.catalog.{CATALOG}.warehouse",        WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG}.glue.id",          GLUE_CATALOG_ID)
    .config(f"spark.sql.catalog.{CATALOG}.glue.endpoint",    GLUE_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG}.client.region",    GLUE_REGION)
    # NOTE: Glue client credentials are NOT set via spark.sql.catalog.X.client.*
    # — Iceberg's Java GlueCatalog only supports that property pair for the
    # S3 client (see https://github.com/apache/iceberg/issues/10614).
    # Glue auth comes from AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY env vars,
    # which were set in the previous cell and are picked up automatically by
    # the AWS SDK's EnvironmentVariableCredentialsProvider.

    # ── Native S3 (Iceberg table DATA) — Dell ObjectScale ────────────────
    # These work as catalog-scoped properties for Iceberg's S3FileIO.
    .config(f"spark.sql.catalog.{CATALOG}.s3.endpoint",          S3_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG}.s3.access-key-id",     S3_ACCESS_KEY)
    .config(f"spark.sql.catalog.{CATALOG}.s3.secret-access-key", S3_SECRET_KEY)
    .config(f"spark.sql.catalog.{CATALOG}.s3.path-style-access", "true")

    # ── Hadoop S3A (streaming CHECKPOINT metadata) — Option B ────────────
    # Spark Structured Streaming's checkpoint mechanism goes through
    # Hadoop's generic FileSystem layer, NOT Iceberg's S3FileIO — so it
    # needs its own, separate S3 config via the standard fs.s3a.* keys,
    # which the hadoop-aws JAR (added to --jars in the previous cell)
    # implements. Same Dell ObjectScale endpoint/creds as above, just a
    # different config namespace because it's a different code path.
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint",            S3_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",           S3_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",           S3_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",    "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")  # endpoint is http://
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

    # ── Tuning (dev scale) ──────────────────────────────────────
    .config("spark.sql.shuffle.partitions",            "4")
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")

    # per-executor resources
    .config("spark.executor.memory",   "4g")    # heap per executor
    .config("spark.executor.cores",    "2")     # cores per executor
    .config("spark.executor.instances","4")     # number of executors (static)
    # driver
    .config("spark.driver.memory",     "2g")
    .config("spark.driver.cores",      "1")
    # off-heap overhead (shuffle, network buffers) — often needed
    .config("spark.executor.memoryOverhead", "1g")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

## Cell 2 — Define Schemas

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, DoubleType,
    BooleanType, LongType, ArrayType
)

# ── Schema 1: Payment transactions (payments.raw) ────────────────────────────
payment_schema = StructType([
    StructField("event_id",          StringType(),    True),
    StructField("event_type",        StringType(),    True),
    StructField("timestamp",         TimestampType(), True),
    StructField("transaction_id",    StringType(),    True),
    StructField("account_id",        StringType(),    True),
    StructField("amount",            DoubleType(),    True),
    StructField("currency",          StringType(),    True),
    StructField("channel",           StringType(),    True),
    StructField("merchant_category", StringType(),    True),
    StructField("status",            StringType(),    True),
    StructField("city",              StringType(),    True),
    StructField("country",           StringType(),    True),
    StructField("latitude",          DoubleType(),    True),
    StructField("longitude",         DoubleType(),    True),
    StructField("is_international",  BooleanType(),   True),
    StructField("customer_tier",     StringType(),    True),
    StructField("risk_score",        DoubleType(),    True),
])

# ── Schema 2: Fraud signals (fraud.signals) ──────────────────────────────────
fraud_schema = StructType([
    StructField("event_id",      StringType(),                       True),
    StructField("event_type",    StringType(),                       True),
    StructField("timestamp",     TimestampType(),                    True),
    StructField("account_id",    StringType(),                       True),
    StructField("fraud_type",    StringType(),                       True),
    StructField("ml_score",      DoubleType(),                       True),
    StructField("rule_triggers", ArrayType(StringType()),            True),
    StructField("action",        StringType(),                       True),
    StructField("case_id",       StringType(),                       True),
])

# ── Schema 3: Customer events (customer.events) ──────────────────────────────
# Union schema covering LOGIN, AML_ALERT, PROFILE_CHANGE, ACCOUNT_OPEN
customer_schema = StructType([
    StructField("event_id",        StringType(),    True),
    StructField("event_type",      StringType(),    True),
    StructField("timestamp",       TimestampType(), True),
    StructField("account_id",      StringType(),    True),
    # AML fields
    StructField("alert_type",      StringType(),    True),
    StructField("risk_band",       StringType(),    True),
    StructField("sar_required",    BooleanType(),   True),
    StructField("total_amount_7d", DoubleType(),    True),
    # Login fields
    StructField("channel",         StringType(),    True),
    StructField("auth_method",     StringType(),    True),
    StructField("login_success",   BooleanType(),   True),
    StructField("new_device",      BooleanType(),   True),
])

print("✅ Schemas defined:")
print(f"  payment_schema  — {len(payment_schema.fields)} fields")
print(f"  fraud_schema    — {len(fraud_schema.fields)} fields")
print(f"  customer_schema — {len(customer_schema.fields)} fields")

## Cell 3 — Verify Existing Iceberg Tables

In [ ]:
# Your Iceberg catalog/schema already exist — this cell just confirms
# connectivity and lists the 5 tables you're streaming into.
# (No CREATE statements needed since the tables are already provisioned.)

EXPECTED_TABLES = {
    "aml_alerts", "customer_events", "fraud_alerts",
    "payment_agg", "payment_transactions",
}

print(f"📋 Tables in {CATALOG}.{DB}:\n")
tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{DB}")
tables_df.show(truncate=False)

found = {row.tableName for row in tables_df.collect()}
missing = EXPECTED_TABLES - found

if missing:
    print(f"⚠️  Missing tables: {missing}")
else:
    print("✅ All 5 expected tables found.")

# Quick schema check on the main fact table
print(f"\n🔍 Schema for {CATALOG}.{DB}.payment_transactions:")
spark.table(f"{CATALOG}.{DB}.payment_transactions").printSchema()

## Cell 4 — Stream 1: Raw Payment Transactions → Iceberg

In [ ]:
import glob
jars = sorted(glob.glob("/home/jovyan/local_jars/*.jar"))
hadoop_aws = [j for j in jars if "hadoop-aws" in j]
aws_sdk = [j for j in jars if "aws-java-sdk-bundle" in j]
print("hadoop-aws found:", hadoop_aws)
print("aws-java-sdk-bundle found:", aws_sdk)
print(f"\nTotal jars: {len(jars)}")

In [ ]:
import os
args = os.environ.get("PYSPARK_SUBMIT_ARGS", "NOT SET")
print("hadoop-aws in args:", "hadoop-aws" in args)
print("aws-java-sdk-bundle in args:", "aws-java-sdk-bundle" in args)

In [ ]:
print(spark.sparkContext.getConf().get("spark.jars", "NOT SET"))

In [ ]:
from pyspark.sql.functions import (
    col, from_json, current_timestamp, to_date,
    window, avg, sum, count, max, when
)

# ── Read from Kafka: payments.raw ────────────────────────────────────────────
payments_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKERS)
    .option("subscribe",               "payments.raw")
    .option("startingOffsets",          "latest")
    .option("maxOffsetsPerTrigger",     5_000)      # dev-friendly back-pressure
    .option("failOnDataLoss",           "false")
    .load()
)

# ── Parse JSON payload ───────────────────────────────────────────────────────
pay_stream = (
    payments_raw
    .select(from_json(col("value").cast("string"), payment_schema).alias("d"))
    .select("d.*")
    .filter(col("event_id").isNotNull())            # drop malformed records
    .withColumn("ingested_at", current_timestamp())
    .withColumn("event_date",  to_date(col("timestamp")))
)

# ── Write raw events to Iceberg ──────────────────────────────────────────────
###
#q_payments = (
#    pay_stream.writeStream
#    .format("iceberg")
#    .outputMode("append")
#    .trigger(processingTime="30 seconds")
#    .option("path",              f"{CATALOG}.{DB}.payment_transactions")
#    .option("checkpointLocation", f"{CHECKPOINT}/payment_transactions")
#    .option("fanout-enabled",    "true")            # allows out-of-order partition writes
#    .start()
#)
###

q_payments = (
    pay_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .queryName("payments_to_iceberg")          # so name isn't None
    .trigger(processingTime="30 seconds")
    .option("checkpointLocation", "s3://aidp-s3ds/warehouse/banking/checkpoints/payment_transactions_v3")
    .option("fanout-enabled", "true")
    .toTable("banking_iceberg.banking.payment_transactions")
)
print(f"✅ Stream started  : {q_payments.name}")
print(f"   Status         : {q_payments.status['message']}")
print(f"   Target table   : {CATALOG}.{DB}.payment_transactions")

In [ ]:
import time, json
time.sleep(40)                      # let 1–2 triggers fire

print("active   :", q_payments.isActive)
print("exception:", q_payments.exception())
print(json.dumps(q_payments.lastProgress, indent=2))

## Cell 5 — Stream 2: Windowed Payment Aggregates → Iceberg

In [ ]:
# ── 5-minute tumbling window aggregates ──────────────────────────────────────
# pay_stream is reused from Cell 4 — the same parsed DataFrame
# can fan out to multiple sinks without re-reading Kafka.

pay_agg = (
    pay_stream
    .withWatermark("timestamp", "2 minutes")        # tolerate 2-min late arrivals
    .groupBy(
        window(col("timestamp"), "5 minutes"),      # 5-min tumbling window
        col("city"),
        col("channel"),
        col("merchant_category"),
    )
    .agg(
        count("*")                              .alias("txn_count"),
        sum("amount")                           .alias("total_amount"),
        avg("amount")                           .alias("avg_amount"),
        max("amount")                           .alias("max_amount"),
        sum(when(col("status") == "APPROVED", 1).otherwise(0)).alias("approved_count"),
        sum(when(col("status") == "DECLINED", 1).otherwise(0)).alias("declined_count"),
        sum(when(col("is_international"),      1).otherwise(0)).alias("intl_count"),
        avg("risk_score")                       .alias("avg_risk_score"),
    )
    # Flatten the window struct into start/end columns
    .select(
        col("window.start") .alias("window_start"),
        col("window.end")   .alias("window_end"),
        col("city"),
        col("channel"),
        col("merchant_category"),
        col("txn_count"),
        col("total_amount"),
        col("avg_amount"),
        col("max_amount"),
        col("approved_count"),
        col("declined_count"),
        col("intl_count"),
        col("avg_risk_score"),
        to_date(col("window.start")).alias("event_date"),
    )
)

q_pay_agg = (
    pay_agg.writeStream
    .format("iceberg")
    .outputMode("append")                           # watermark required for append mode
    .trigger(processingTime="60 seconds")
    .option("path",              f"{CATALOG}.{DB}.payment_agg")
    .option("checkpointLocation", f"{CHECKPOINT}/payment_agg")
    .start()
)

print(f"✅ Stream started  : {q_pay_agg.name}")
print(f"   Status         : {q_pay_agg.status['message']}")
print(f"   Target table   : {CATALOG}.{DB}.payment_agg")

## Cell 6 — Stream 3: Fraud Signals + AML + Customer Events → Iceberg

This stream reads `fraud.signals` and `customer.events` together using `subscribePattern`,  
then routes each event type to a different Iceberg table inside `foreachBatch`.

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import lit

# ── Combined schema for all non-payment events ───────────────────────────────
combined_schema = StructType([
    StructField("event_id",        StringType(),    True),
    StructField("event_type",      StringType(),    True),
    StructField("timestamp",       TimestampType(), True),
    StructField("account_id",      StringType(),    True),
    # fraud fields
    StructField("fraud_type",      StringType(),    True),
    StructField("ml_score",        DoubleType(),    True),
    StructField("action",          StringType(),    True),
    StructField("case_id",         StringType(),    True),
    # aml fields
    StructField("alert_type",      StringType(),    True),
    StructField("risk_band",       StringType(),    True),
    StructField("sar_required",    BooleanType(),   True),
    StructField("total_amount_7d", DoubleType(),    True),
    # login fields
    StructField("channel",         StringType(),    True),
    StructField("auth_method",     StringType(),    True),
    StructField("login_success",   BooleanType(),   True),
    StructField("new_device",      BooleanType(),   True),
])

# ── Read two topics at once via comma-separated subscribe ────────────────────
events_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKERS)
    .option("subscribe",               "fraud.signals,customer.events")
    .option("startingOffsets",          "latest")
    .option("maxOffsetsPerTrigger",     2_000)
    .option("failOnDataLoss",           "false")
    .load()
    .select(from_json(col("value").cast("string"), combined_schema).alias("d"))
    .select("d.*")
    .withColumn("event_date", to_date(col("timestamp")))
)

# ── foreachBatch: route each event type to a separate Iceberg table ──────────
def route_to_iceberg(batch_df: DataFrame, batch_id: int) -> None:
    """Called once per micro-batch. Routes events to the correct Iceberg table."""

    if batch_df.isEmpty():
        return

    # Cache — we'll scan the batch multiple times for different routes
    batch_df.cache()

    # ── Route 1: Fraud alerts ────────────────────────────────────────────────
    fraud_df = (
        batch_df
        .filter(col("event_type") == "FRAUD_SIGNAL")
        .select(
            "event_id", "event_type", "timestamp", "account_id",
            "fraud_type", "ml_score", "action", "case_id", "event_date"
        )
    )
    if not fraud_df.isEmpty():
        fraud_df.writeTo(f"{CATALOG}.{DB}.fraud_alerts").append()
        print(f"  [batch {batch_id}] fraud_alerts  ← {fraud_df.count()} rows")

    # ── Route 2: AML alerts ──────────────────────────────────────────────────
    aml_df = (
        batch_df
        .filter(col("event_type") == "AML_ALERT")
        .select(
            "event_id", "event_type", "timestamp", "account_id",
            "alert_type", "risk_band", "sar_required", "total_amount_7d", "event_date"
        )
    )
    if not aml_df.isEmpty():
        aml_df.writeTo(f"{CATALOG}.{DB}.aml_alerts").append()
        print(f"  [batch {batch_id}] aml_alerts    ← {aml_df.count()} rows")

    # ── Route 3: Customer login / profile / account events ───────────────────
    customer_df = (
        batch_df
        .filter(col("event_type").isin(
            "CUSTOMER_LOGIN", "PROFILE_CHANGE", "ACCOUNT_OPEN"
        ))
        .select(
            "event_id", "event_type", "timestamp", "account_id",
            "channel", "auth_method", "login_success", "new_device", "event_date"
        )
    )
    if not customer_df.isEmpty():
        customer_df.writeTo(f"{CATALOG}.{DB}.customer_events").append()
        print(f"  [batch {batch_id}] customer_events← {customer_df.count()} rows")

    batch_df.unpersist()


q_events = (
    events_raw.writeStream
    .foreachBatch(route_to_iceberg)
    .trigger(processingTime="30 seconds")
    .option("checkpointLocation", f"{CHECKPOINT}/fraud_customer")
    .start()
)

print(f"✅ Stream started  : {q_events.name}")
print(f"   Status         : {q_events.status['message']}")
print(f"   Topics         : fraud.signals, customer.events")
print(f"   Targets        : fraud_alerts, aml_alerts, customer_events")

## Cell 7 — Monitor Active Streams

In [ ]:
import time

active = spark.streams.active
print(f"Active streaming queries: {len(active)}\n")

for q in active:
    prog = q.lastProgress
    if prog:
        rows_in  = prog.get("numInputRows", 0)
        rows_sec = prog.get("processedRowsPerSecond", 0.0)
        trigger  = prog.get("trigger", {}).get("processingTime", "?")
        print(
            f"  [{q.name}]\n"
            f"    Status     : {q.status['message']}\n"
            f"    Rows in    : {rows_in:,}\n"
            f"    Throughput : {rows_sec:.1f} rows/sec\n"
            f"    Batch time : {trigger}\n"
        )
    else:
        print(f"  [{q.name}] — waiting for first batch...\n")

In [ ]:
# ── Poll progress every 10 seconds for 2 minutes (useful for demo runs) ──────
# Interrupt the kernel (■) to stop early.

POLL_SECONDS = 120
INTERVAL     = 10

queries = {"q_payments": q_payments, "q_pay_agg": q_pay_agg, "q_events": q_events}

for i in range(POLL_SECONDS // INTERVAL):
    print(f"\n── tick {i+1} ({(i+1)*INTERVAL}s) {'─'*40}")
    for name, q in queries.items():
        if q.isActive:
            p = q.lastProgress or {}
            print(f"  {name:15s} | rows={p.get('numInputRows',0):>6,} "
                  f"| {p.get('processedRowsPerSecond',0.0):>8.1f} rows/s "
                  f"| {q.status['message']}")
        else:
            print(f"  {name:15s} | STOPPED")
    time.sleep(INTERVAL)

## Cell 8 — Ad-hoc Iceberg Queries

These run against data already written to Iceberg — you can execute them while streams are running.

In [ ]:
# ── Q1: Transaction approval rate by channel (last 5 minutes) ────────────────
spark.sql(f"""
    SELECT
        channel,
        COUNT(*) AS total_txns,
        SUM(CASE WHEN status = 'APPROVED' THEN 1 ELSE 0 END) AS approved,
        ROUND(AVG(CASE WHEN status = 'APPROVED' THEN 1.0 ELSE 0.0 END) * 100, 2) AS approval_rate_pct,
        ROUND(SUM(amount), 2)    AS total_value,
        ROUND(AVG(risk_score), 4) AS avg_risk
    FROM {CATALOG}.{DB}.payment_transactions
    WHERE timestamp >= current_timestamp() - INTERVAL 5 MINUTES
    GROUP BY channel
    ORDER BY total_txns DESC
""").show(truncate=False)

In [ ]:
# ── Q2: Top merchant categories by spend today ───────────────────────────────
spark.sql(f"""
    SELECT
        merchant_category,
        COUNT(*) AS txn_count,
        ROUND(SUM(amount), 2) AS total_spend,
        ROUND(AVG(amount), 2) AS avg_txn_value
    FROM {CATALOG}.{DB}.payment_transactions
    WHERE event_date = current_date()
      AND status = 'APPROVED'
    GROUP BY merchant_category
    ORDER BY total_spend DESC
""").show(truncate=False)

In [ ]:
# ── Q3: Fraud alerts in the last hour ────────────────────────────────────────
spark.sql(f"""
    SELECT
        fraud_type,
        action,
        COUNT(*) AS alert_count,
        ROUND(AVG(ml_score), 3) AS avg_ml_score
    FROM {CATALOG}.{DB}.fraud_alerts
    WHERE timestamp >= current_timestamp() - INTERVAL 1 HOUR
    GROUP BY fraud_type, action
    ORDER BY alert_count DESC
""").show(truncate=False)

In [ ]:
# ── Q4: AML cases requiring SAR (today, HIGH risk) ───────────────────────────
spark.sql(f"""
    SELECT account_id, alert_type, risk_band, total_amount_7d, timestamp
    FROM {CATALOG}.{DB}.aml_alerts
    WHERE event_date = current_date()
      AND sar_required = true
      AND risk_band = 'HIGH'
    ORDER BY total_amount_7d DESC
""").show(20, truncate=False)

In [ ]:
# ── Q5: Failed logins in last hour (potential account takeover) ───────────────
spark.sql(f"""
    SELECT
        account_id,
        COUNT(*) AS failed_logins,
        MIN(timestamp) AS first_attempt,
        MAX(timestamp) AS last_attempt
    FROM {CATALOG}.{DB}.customer_events
    WHERE event_type = 'CUSTOMER_LOGIN'
      AND login_success = false
      AND timestamp >= current_timestamp() - INTERVAL 1 HOUR
    GROUP BY account_id
    HAVING COUNT(*) >= 3
    ORDER BY failed_logins DESC
""").show(20, truncate=False)

In [ ]:
# ── Q6: Iceberg time-travel — read payment_transactions as of 1 hour ago ─────
from datetime import datetime, timedelta, timezone

one_hour_ago = (datetime.now(timezone.utc) - timedelta(hours=1)).strftime("%Y-%m-%d %H:%M:%S")

(
    spark.read
    .option("as-of-timestamp", str(int((datetime.now(timezone.utc) - timedelta(hours=1)).timestamp() * 1000)))
    .format("iceberg")
    .load(f"{CATALOG}.{DB}.payment_transactions")
    .orderBy(col("timestamp").desc())
    .limit(10)
    .show(truncate=False)
)

In [ ]:
# ── Q7: Iceberg snapshot history ─────────────────────────────────────────────
spark.sql(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM {CATALOG}.{DB}.payment_transactions.snapshots
    ORDER BY committed_at DESC
    LIMIT 10
""").show(truncate=False)

# ── Q8: Table file stats ──────────────────────────────────────────────────────
spark.sql(f"""
    SELECT
        partition,
        record_count,
        file_count,
        total_data_file_size_in_bytes / 1024 / 1024 AS size_mb
    FROM {CATALOG}.{DB}.payment_transactions.partitions
    ORDER BY record_count DESC
""").show(20, truncate=False)

In [ ]:
# ── Q9: Compact small files (run periodically) ────────────────────────────────
from pyspark.sql.functions import current_date

today = datetime.now().strftime("%Y-%m-%d")

spark.sql(f"""
    CALL {CATALOG}.system.rewrite_data_files(
        table => '{DB}.payment_transactions',
        strategy => 'binpack',
        options => map(
            'target-file-size-bytes', '134217728',
            'min-input-files', '3'
        ),
        where => 'event_date = \'{today}\''
    )
""").show()

# ── Q10: Expire old snapshots (keep last 3 days) ──────────────────────────────
expire_ts = (datetime.now(timezone.utc) - timedelta(days=3)).strftime("%Y-%m-%d %H:%M:%S")

spark.sql(f"""
    CALL {CATALOG}.system.expire_snapshots(
        table => '{DB}.payment_transactions',
        older_than => TIMESTAMP '{expire_ts}',
        retain_last => 5
    )
""").show()

## Cell 9 — Stop All Streams

In [ ]:
# Graceful stop — flushes in-flight data before shutting down.
# Run this cell when you're done with the demo.

for name, q in {"q_payments": q_payments, "q_pay_agg": q_pay_agg, "q_events": q_events}.items():
    if q.isActive:
        q.stop()
        print(f"⏹  Stopped : {name}")
    else:
        print(f"   Already stopped : {name}")

print(f"\nActive queries remaining: {len(spark.streams.active)}")